In [ ]:
import json
import math
from collections import defaultdict

# results/clinvar_location_surface.json を読み込み
json_open = open("../results/clinvar_location_surface.json", "r")
data = json.load(json_open)

# makeHeatmap_0327 と同じアミノ酸順
aa = ['Ile', 'Val', 'Leu', 'Phe', 'Cys', 'Met', 'Ala', 'Gly', 'Thr', 'Ser', 'Trp', 'Tyr', 'Pro', 'His', 'Glu', 'Gln', 'Asp', 'Asn', 'Lys', 'Arg']

# カテゴリ一覧: "Cytoplasm-buried", "Cytoplasm-surface", ...
categories = []
for location in data:
    for surface_type in data[location]:
        if isinstance(data[location][surface_type], dict):
            categories.append(f"{location}-{surface_type}")

# カテゴリごとに disease / polymorphism を集計
# キーは "Before->After" 形式なので before+after に変換して扱う
disease = {}
polymorphism = {}
for cat in categories:
    disease[cat] = defaultdict(int)
    polymorphism[cat] = defaultdict(int)

for location in data:
    for surface_type in data[location]:
        sub = data[location][surface_type]
        if not isinstance(sub, dict):
            continue
        cat = f"{location}-{surface_type}"
        for aa_change, counts in sub.items():
            if "->" not in aa_change:
                continue
            before, after = aa_change.split("->", 1)
            key = before + after  # makeHeatmap_0327 と同じ形式
            d = counts.get("Disease", 0)
            p = counts.get("Polymorphism", 0)
            disease[cat][key] += d
            polymorphism[cat][key] += p

# 局在を考えずに全カテゴリを合算して wo_localize (All の d/(d+p)) を計算
wo_localize = {}
for b in aa:
    wo_localize[b] = {}
    for a in aa:
        var = b + a
        d_total = sum(disease[c].get(var, 0) for c in categories)
        p_total = sum(polymorphism[c].get(var, 0) for c in categories)
        if d_total + p_total != 0:
            wo_localize[b][a] = d_total / (d_total + p_total)
        else:
            wo_localize[b][a] = "-"

In [ ]:
# 各カテゴリ（Cytoplasm-buried, Cytoplasm-surface, ...）ごとにヒートマップ用の行列を出力
# 形式は makeHeatmap_0327 と同じ: log((d/(d+p)) / wo_localize[b][a])
for cat in categories:
    print("{}\t{}".format(cat, "\t".join(aa)))
    for b in aa:
        print("{}\t".format(b), end="")
        for a in aa:
            var = b + a
            d = disease[cat].get(var, 0)
            p = polymorphism[cat].get(var, 0)
            if d + p != 0 and wo_localize[b][a] != "-":
                try:
                    val = (d / (d + p)) / wo_localize[b][a]
                    if val > 0:
                        print("{}\t".format(math.log(val)), end="")
                    else:
                        print("-\t", end="")
                except (ValueError, ZeroDivisionError):
                    print("-\t", end="")
            else:
                print("-\t", end="")
        print()
    print()